# Data Science Project: Wine Quality Analysis

## Overview
This notebook demonstrates a complete data science workflow including:
- Data loading and preprocessing
- Exploratory data analysis (EDA)
- Data visualization
- Feature engineering
- Machine learning model development
- Model evaluation

## Dataset
We'll use a synthetic wine quality dataset for this analysis.

## 1. Import Libraries

In [ ]:
# Data manipulation
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set random seed for reproducibility
np.random.seed(42)

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 2. Data Generation and Loading

For demonstration purposes, we'll generate a synthetic wine quality dataset with realistic features.

In [ ]:
# Generate synthetic wine dataset
n_samples = 1000

# Create features
data = {
    'fixed_acidity': np.random.normal(7.0, 1.5, n_samples),
    'volatile_acidity': np.random.normal(0.5, 0.2, n_samples),
    'citric_acid': np.random.normal(0.3, 0.15, n_samples),
    'residual_sugar': np.random.exponential(2.5, n_samples),
    'chlorides': np.random.normal(0.08, 0.02, n_samples),
    'free_sulfur_dioxide': np.random.normal(30, 10, n_samples),
    'total_sulfur_dioxide': np.random.normal(100, 30, n_samples),
    'density': np.random.normal(0.996, 0.002, n_samples),
    'pH': np.random.normal(3.3, 0.2, n_samples),
    'sulphates': np.random.normal(0.6, 0.15, n_samples),
    'alcohol': np.random.normal(10.5, 1.5, n_samples)
}

df = pd.DataFrame(data)

# Create quality target based on features (simplified logic)
quality_score = (
    0.3 * df['alcohol'] +
    -2.0 * df['volatile_acidity'] +
    0.5 * df['citric_acid'] +
    np.random.normal(0, 1, n_samples)
)

# Convert to quality categories (0: Low, 1: Medium, 2: High)
df['quality'] = pd.cut(quality_score, bins=3, labels=[0, 1, 2])
df['quality'] = df['quality'].astype(int)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("Dataset Information:")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nBasic statistics:\n{df.describe()}")

In [ ]:
# Quality distribution
print("Quality Distribution:")
print(df['quality'].value_counts().sort_index())
print(f"\nQuality proportions:")
print(df['quality'].value_counts(normalize=True).sort_index())

## 4. Data Visualization

In [ ]:
# Quality distribution bar plot
plt.figure(figsize=(10, 6))
quality_counts = df['quality'].value_counts().sort_index()
plt.bar(quality_counts.index, quality_counts.values, color=['red', 'orange', 'green'])
plt.xlabel('Quality Category', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Distribution of Wine Quality', fontsize=14, fontweight='bold')
plt.xticks([0, 1, 2], ['Low', 'Medium', 'High'])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
features_to_plot = ['alcohol', 'volatile_acidity', 'citric_acid', 'pH', 'sulphates', 'residual_sugar']

for idx, feature in enumerate(features_to_plot):
    row = idx // 3
    col = idx % 3
    axes[row, col].hist(df[feature], bins=30, edgecolor='black', alpha=0.7)
    axes[row, col].set_xlabel(feature.replace('_', ' ').title(), fontsize=10)
    axes[row, col].set_ylabel('Frequency', fontsize=10)
    axes[row, col].set_title(f'Distribution of {feature.replace("_", " ").title()}', fontsize=11)
    axes[row, col].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for features by quality
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
features_to_plot = ['alcohol', 'volatile_acidity', 'citric_acid', 'pH', 'sulphates', 'fixed_acidity']

for idx, feature in enumerate(features_to_plot):
    row = idx // 3
    col = idx % 3
    df.boxplot(column=feature, by='quality', ax=axes[row, col])
    axes[row, col].set_xlabel('Quality Category', fontsize=10)
    axes[row, col].set_ylabel(feature.replace('_', ' ').title(), fontsize=10)
    axes[row, col].set_title(f'{feature.replace("_", " ").title()} by Quality', fontsize=11)
    axes[row, col].set_xticklabels(['Low', 'Medium', 'High'])

plt.suptitle('Feature Distributions by Quality Category', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 5. Feature Engineering and Preparation

In [ ]:
# Separate features and target
X = df.drop('quality', axis=1)
y = df['quality']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"\nTraining set quality distribution:\n{y_train.value_counts().sort_index()}")
print(f"\nTesting set quality distribution:\n{y_test.value_counts().sort_index()}")

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")
print(f"\nMean of scaled training features: {X_train_scaled.mean(axis=0).round(2)}")
print(f"Std of scaled training features: {X_train_scaled.std(axis=0).round(2)}")

## 6. Model Development

In [ ]:
# Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, min_samples_split=5)
rf_model.fit(X_train_scaled, y_train)

print("Random Forest model trained successfully!")
print(f"Number of trees: {rf_model.n_estimators}")
print(f"Max depth: {rf_model.max_depth}")

## 7. Model Evaluation

In [ ]:
# Make predictions
y_train_pred = rf_model.predict(X_train_scaled)
y_test_pred = rf_model.predict(X_test_scaled)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print(f"\nOverfitting check: {train_accuracy - test_accuracy:.4f}")

In [ ]:
# Classification report
print("Classification Report (Test Set):")
print(classification_report(y_test, y_test_pred, target_names=['Low', 'Medium', 'High']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True, 
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'])
plt.xlabel('Predicted Quality', fontsize=12)
plt.ylabel('Actual Quality', fontsize=12)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Feature Importance Analysis

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

In [ ]:
# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance in Wine Quality Prediction', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Conclusions

### Key Findings:
1. **Model Performance**: The Random Forest classifier achieved good accuracy in predicting wine quality categories.
2. **Important Features**: The most important features for predicting wine quality include alcohol content, volatile acidity, and citric acid.
3. **Data Patterns**: Clear patterns exist between chemical properties and wine quality.

### Next Steps:
- Try other algorithms (e.g., Gradient Boosting, SVM, Neural Networks)
- Perform hyperparameter tuning using GridSearchCV or RandomizedSearchCV
- Collect more data to improve model generalization
- Engineer additional features (e.g., ratios, interactions)
- Deploy the model as a web service for real-time predictions

### Technical Notes:
- Data was synthetically generated for demonstration purposes
- Model was trained with standard parameters
- Cross-validation could provide better performance estimates
- Feature scaling improved model convergence